# Visual Introduction to RNNs (PyTorch)

This notebook helps visualize how an RNN processes sequences and how the hidden state evolves.

Objetivos:
- Entender cómo se representa una secuencia en PyTorch (batch, seq_len, features).
- Implementar una RNN manual para inspeccionar estados ocultos y visualizar su evolución.
- Entrenar un clasificador sencillo que detecte un patrón en la secuencia.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Create a Toy Sequence Task

Task: clasificar secuencias donde aparece un patrón simple (3 valores altos).

In [ ]:
def generate_data(batch_size=64, seq_length=30):
    X = torch.randn(batch_size, seq_length, 1)
    y = torch.zeros(batch_size, dtype=torch.long)

    # Inject pattern in half the sequences
    for i in range(batch_size // 2):
        pos = torch.randint(5, seq_length-5, (1,)).item()
        X[i, pos:pos+3] += 3  # simple pattern
        y[i] = 1

    return X, y

# quick sanity check
X, y = generate_data()
print(X.shape, y.shape)

## 2. Define a Custom RNN (to access hidden states)

Implementamos una RNN manual para poder devolver los estados ocultos en cada paso de tiempo y visualizarlos.

In [ ]:
class SimpleRNNManual(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.Wx = nn.Linear(input_size, hidden_size)
        self.Wh = nn.Linear(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        batch_size, seq_len, _ = x.shape
        # initialize hidden state on the same device as input
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        hidden_states = []

        for t in range(seq_len):
            h = torch.tanh(self.Wx(x[:, t, :]) + self.Wh(h))
            hidden_states.append(h.unsqueeze(1))

        hidden_states = torch.cat(hidden_states, dim=1)
        out = self.fc(h)

        return out, hidden_states

In [ ]:
# instantiate model, loss and optimizer
model = SimpleRNNManual(input_size=1, hidden_size=8, output_size=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 3. Train the Model

Entrenamos un clasificador sencillo durante varias épocas y mostramos pérdida periódicamente.

In [ ]:
for epoch in range(50):
    X, y = generate_data()

    outputs, _ = model(X)
    loss = criterion(outputs, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

## 4. Visualize Hidden State Evolution

Tomamos una secuencia de prueba, calculamos los estados ocultos y los mostramos en una imagen (cada fila = neurona oculta, columnas = tiempo).

In [ ]:
X_test, y_test = generate_data(batch_size=1)
output, hidden_states = model(X_test)

hidden_states = hidden_states.detach().numpy()[0]  # (seq_len, hidden_size)
input_signal = X_test.detach().numpy()[0, :, 0]

plt.figure(figsize=(12,5))
plt.subplot(2,1,1)
plt.plot(input_signal)
plt.title("Input Sequence")

plt.subplot(2,1,2)
plt.imshow(hidden_states.T, aspect='auto', cmap='viridis')
plt.colorbar()
plt.title("Hidden State Evolution (each row = neuron)")
plt.xlabel("Time")
plt.ylabel("Hidden Units")

plt.tight_layout()
plt.show()

## 5. Interpretation

- Cada fila en la visualización de estados ocultos corresponde a una neurona oculta.
- Las columnas son pasos de tiempo.
- El patrón inyectado activa diferentes neuronas cuando aparece.

## 6. Questions for Students
- ¿Qué pasa si aumentamos la longitud de la secuencia?
- ¿Recuerda la red patrones tempranos cuando la secuencia es larga?
- ¿Qué sucede si quitamos la función tanh?
- ¿Qué pasa si aumentamos el tamaño del estado oculto?